# DBN (2006) — 노트가 남긴 빈칸을 직접 재 본다

Hinton, Osindero, Teh 의 **탐욕 층별 학습**을 numpy 로 구현해, DBN 노트의 「내가 짚은 자리」와 「돌려 볼 수 있는 것」에 답이 나오는지 본다.

| 실험 | 묻는 것 | 노트의 어느 자리에 닿는가 |
|---|---|---|
| A | RBM 하나가 실제로 배워지는가. 필터가 획처럼 생기는가 | 20절 둘 — 이 논문의 알맹이 중 실제로 도는 최소 단위 |
| B | 대조 발산 걸음 수 `n` 을 늘리면 무엇이 좋아지고 무엇이 비싸지는가 | 16절 셋 — 논문에 `n` 을 바꿔 본 표가 없다 |
| C | 사전학습한 초기화에서는 층별 기울기가 덜 줄어드는가 | 20절 하나 — 역전파 실험 E(층마다 10~12배)와 잇는 다리 |
| **D** | **무작위 초기화에 같은 미세조정만 얹으면 얼마가 나오는가. 그리고 층을 깊게 할수록 그 차이가 커지는가** | **16절 하나와 둘 — 노트가 꼽은 가장 큰 빈칸이고 INDEX 의 다음 읽을 것 1순위** |
| E | 라벨이 귀할 때 사전학습의 이점이 커지는가 | 논문 12절 — 「비지도는 라벨 없는 큰 데이터를 쓸 수 있다」의 실측 |

---

## 돌리기 전에 밝혀 두는 것 넷

**하나. 이 노트북은 앞의 둘과 달리 바깥 데이터를 읽는다.** 퍼셉트론·역전파 노트북은 데이터를 스스로 만들었지만 이 논문의 주장은 MNIST 위에 서 있다. `sklearn.datasets.fetch_openml` 로 한 번 받아 `data/` 에 캐시한다(`.gitignore` 가 막으므로 저장소에 올라가지 않는다). 받지 못하면 그 사실을 적고 멈춘다 — **합성 데이터로 바꿔치기하지 않는다.**

**둘. 규모를 줄였다.** 논문은 60,000 장에 784-500-500-2000 을 일주일 배웠다. 여기서는 **학습 10,000 장 · 시험 2,000 장 · 은닉 256** 이다. 줄인 이유는 numpy·CPU 로 몇십 분 안에 끝내려는 것이고, **줄인 채로도 D 의 비교는 성립한다** — 두 조건이 같은 데이터·같은 크기·같은 미세조정을 쓰기 때문이다. 다만 **절대 오차율을 논문의 1.25% 와 같은 칸에 놓지 않는다.**

**셋. 미세조정이 논문의 것과 다르다.** 논문의 위-아래 알고리즘은 **생성** 모델을 다듬는 절차다. 여기서 쓰는 것은 **지도 역전파**다. 바꾼 이유는 D 가 묻는 것이 「사전학습이 뒤에 오는 학습을 돕는가」이고, 그 물음은 미세조정을 고정해 두고 초기화만 바꿔야 답이 나오기 때문이다. **그래서 D 의 결과는 「논문의 파이프라인에 탐욕 단계가 필요했나」에 직접 답하지 않는다.** 답하는 것은 「비지도 사전학습이 지도 미세조정을 돕는가」다. 이 구분을 결과에 적는다.

**넷. 축소 규칙을 미리 적는다.** 실험 하나가 10 분을 넘으면 `SEEDS` 를 절반으로 줄인다. **에폭 수와 층 크기는 줄이지 않는다** — 줄이면 「사전학습이 필요 없었다」와 「둘 다 덜 배웠다」가 섞인다.

---

## 가설 — 돌리기 전에 적는다

- **H1 (A)**: CD-1 로 배운 RBM 의 재구성 오차가 에폭을 따라 내려가고, 가중치 필터에 **획이나 얼룩 모양**이 보인다.
  - **H1 반대 방향**: 오차가 안 내려가거나 필터가 잡음으로 남는다. 그러면 구현이 틀렸거나 학습률이 안 맞는 것이다.
- **H2 (B)**: `n` 을 1 에서 10 으로 올리면 재구성 오차가 내려가고 시간이 거의 `n` 에 비례해 는다.
  - **H2 반대 방향**: 오차가 안 내려간다. 그러면 이 규모에서는 CD-1 로 충분하다는 뜻이고, 논문이 `n=1` 을 쓴 선택을 뒷받침한다.
- **H3 (C)**: 사전학습한 초기화에서 **아래층 기울기가 무작위 초기화보다 크다.**
  - **H3 반대 방향**: 차이가 없다. 그러면 사전학습의 이점이 기울기 크기가 아니라 다른 자리(가중치가 놓인 위치)에 있다는 뜻이다.
- **H4 (D)**: 사전학습한 쪽의 시험 오차가 낮고, **그 차이가 층이 깊어질수록 커진다.**
  - **H4 반대 방향**: 차이가 seed 다섯의 변동 폭 안이다. 그러면 이 규모에서 사전학습이 값을 못 낸 것이고, **그것이 노트 16절의 물음에 대한 답이 된다.** 어느 쪽이 나와도 적을 수 있다.
- **H5 (E)**: 라벨을 100 장으로 줄이면 사전학습의 이점이 10,000 장일 때보다 **크다.**
  - **H5 반대 방향**: 라벨 수와 무관하게 차이가 일정하다.

**판정 규칙.** seed 다섯의 **변동 폭(최소~최대)을 먼저 적고, 그 폭보다 작은 차이로 순서를 매기지 않는다.** 완주하지 못한 조건은 빈칸으로 두지 않고 「돌지 않았다」와 그때의 값을 함께 적는다.

**돌리는 법.** 위에서 아래로 모두 실행한다. CPU 만 쓴다. 결과는 `results/` 에 CSV 로, 그림은 `figures/` 에 PNG 로 떨어지고 맨 아래 칸이 요약을 찍는다.

In [1]:
# ── 준비 ────────────────────────────────────────────────────────────────
import os, sys, time, platform
import numpy as np
import pandas as pd

NOTEBOOK = "dbn_2006.ipynb"
RUN_ID   = time.strftime("%Y%m%d_%H%M%S")
HERE     = os.getcwd()
RESULTS  = os.path.join(HERE, "results")
FIGURES  = os.path.join(HERE, "figures")
DATA     = os.path.join(HERE, "data")
for d in (RESULTS, FIGURES, DATA):
    os.makedirs(d, exist_ok=True)

ENV = {
    "run_id": RUN_ID, "notebook": NOTEBOOK,
    "python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
    "platform": platform.platform(), "cwd": HERE,
}
for k, v in ENV.items():
    print("%-10s %s" % (k, v))


def save(df, name):
    """결과 CSV 에 재현 정보를 열로 붙여 저장한다."""
    df = df.copy()
    for k in ("run_id", "notebook", "python", "numpy", "platform"):
        df[k] = ENV[k]
    path = os.path.join(RESULTS, name)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print("저장:", name, df.shape)
    return df


def span(v):
    """변동 폭을 '최소~최대' 로 적는다. 판정 규칙이 요구하는 것."""
    v = np.asarray(v, float)
    return "%.4f~%.4f" % (v.min(), v.max())


# ── 이 실행의 손잡이 ─────────────────────────────────────────────────────
N_TRAIN   = 10_000      # 논문은 60,000. 줄인 이유는 맨 위에 적었다
N_TEST    = 2_000
HIDDEN    = 256         # 논문은 500-500-2000
MB        = 100
RBM_EPOCH = 15
FT_EPOCH  = 30
SEEDS     = 5
FAST      = False       # True 면 seed 를 절반으로 줄인다(축소 규칙)
if FAST:
    SEEDS = max(2, SEEDS // 2)

print()
print("학습 %d 장 · 시험 %d 장 · 은닉 %d · 묶음 %d" % (N_TRAIN, N_TEST, HIDDEN, MB))
print("RBM %d 에폭 · 미세조정 %d 에폭 · seed %d 개" % (RBM_EPOCH, FT_EPOCH, SEEDS))

run_id     20260920_142811
notebook   dbn_2006.ipynb
python     3.13.15
numpy      2.1.3
pandas     2.2.3
platform   Linux-6.6.122+-x86_64-with-glibc2.39
cwd        /content

학습 10000 장 · 시험 2000 장 · 은닉 256 · 묶음 100
RBM 15 에폭 · 미세조정 30 에폭 · seed 5 개


## 0. 데이터 — MNIST 를 한 번 받아 캐시한다

받지 못하면 **그 사실을 적고 멈춘다.** 합성 데이터로 바꿔치기하면 D 의 결과를 논문 이야기와 잇지 못한다.

픽셀을 0~1 로 나눈다. 논문도 아래층 RBM 의 보이는 유닛을 **0 과 1 사이의 실수값(정규화한 픽셀 세기)** 으로 두었다고 적는다.

In [2]:
# ── MNIST ───────────────────────────────────────────────────────────────
CACHE = os.path.join(DATA, "mnist_784.npz")

if os.path.isfile(CACHE):
    z = np.load(CACHE)
    X_all, y_all = z["X"], z["y"]
    print("캐시에서 읽었다:", CACHE, X_all.shape)
else:
    print("MNIST 를 받는다 (처음 한 번만, 수십 초 걸린다) ...")
    from sklearn.datasets import fetch_openml
    mn = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")
    X_all = (mn.data.astype(np.float32) / 255.0)
    y_all = mn.target.astype(np.int64)
    np.savez_compressed(CACHE, X=X_all, y=y_all)
    print("받아서 캐시했다:", CACHE, X_all.shape)

assert X_all.shape[1] == 784, X_all.shape

_rs = np.random.default_rng(12345)
_perm = _rs.permutation(X_all.shape[0])
tr_idx, te_idx = _perm[:N_TRAIN], _perm[N_TRAIN:N_TRAIN + N_TEST]
Xtr, ytr = X_all[tr_idx].astype(np.float64), y_all[tr_idx]
Xte, yte = X_all[te_idx].astype(np.float64), y_all[te_idx]
Ytr = np.eye(10)[ytr]

print("학습", Xtr.shape, "시험", Xte.shape)
print("픽셀 범위 %.3f ~ %.3f · 평균 %.3f" % (Xtr.min(), Xtr.max(), Xtr.mean()))
print("부류별 장수", np.bincount(ytr, minlength=10))

MNIST 를 받는다 (처음 한 번만, 수십 초 걸린다) ...
받아서 캐시했다: /content/data/mnist_784.npz (70000, 784)
학습 (10000, 784) 시험 (2000, 784)
픽셀 범위 0.000 ~ 1.000 · 평균 0.131
부류별 장수 [ 958 1136 1028 1028 1005  945 1000 1013  943  944]


## 1. 구현 — RBM(CD-k) 과 지도 미세조정

**RBM 한 층.** 식 5 의 학습 규칙을 대조 발산으로 근사한다. 데이터를 고정했을 때의 상관에서, 기브스를 `k` 걸음 돈 뒤의 상관을 뺀다.

$$\Delta w_{ij} \ \propto\ \langle v^0_i h^0_j \rangle - \langle v^k_i h^k_j \rangle$$

논문을 따라 **숨은 층은 확률적 이진값**으로 표집하고, **보이는 층 재구성은 확률값을 그대로** 쓴다(표집 잡음을 줄이는 흔한 방식이다). 위층 RBM 의 보이는 값은 아래층 RBM 의 숨은 유닛 **활성 확률**을 쓴다 — 논문 6.1 절이 그렇게 적는다.

**미세조정.** 로지스틱 은닉층 위에 소프트맥스 출력을 얹고 역전파로 배운다. **논문의 위-아래 알고리즘이 아니다**(맨 위에 밝힌 이유).

In [3]:
# ── 구현 ────────────────────────────────────────────────────────────────
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -60, 60)))


def train_rbm(V, n_hid, epochs, rng, k=1, lr=0.05, mb=MB, momentum=0.5, wd=2e-4, log=None):
    """RBM 한 층을 대조 발산 k 걸음으로 배운다. 반환 (W, bv, bh)."""
    n, d = V.shape
    W = rng.normal(0, 0.01, (d, n_hid))
    bv = np.zeros(d)
    bh = np.zeros(n_hid)
    dW = np.zeros_like(W)
    for ep in range(epochs):
        idx = rng.permutation(n)
        err = 0.0
        for s in range(0, n, mb):
            v0 = V[idx[s:s + mb]]
            m = v0.shape[0]
            ph0 = sigmoid(v0 @ W + bh)
            h = (rng.random(ph0.shape) < ph0).astype(np.float64)   # 숨은 층은 이진 표집
            for _ in range(k):                                     # 기브스 k 걸음
                v = sigmoid(h @ W.T + bv)                          # 보이는 층은 확률값
                ph = sigmoid(v @ W + bh)
                h = (rng.random(ph.shape) < ph).astype(np.float64)
            pos = v0.T @ ph0
            neg = v.T @ ph
            dW = momentum * dW + lr * ((pos - neg) / m - wd * W)
            W += dW
            bv += lr * (v0 - v).mean(axis=0)
            bh += lr * (ph0 - ph).mean(axis=0)
            err += ((v0 - v) ** 2).sum()
        rmse = np.sqrt(err / V.size)
        if log is not None:
            log.append((ep + 1, rmse))
    return W, bv, bh


def rbm_up(V, W, bh):
    """아래층의 활성 확률을 위층의 '데이터' 로 쓴다 (논문 6.1 절)."""
    return sigmoid(V @ W + bh)


def pretrain_stack(V, sizes, epochs, rng, k=1):
    """탐욕 층별 학습 — 한 층 배우고 얼리고 그 표현 위에서 다음 층."""
    Ws, bs, cur = [], [], V
    for h in sizes:
        W, _bv, bh = train_rbm(cur, h, epochs, rng, k=k)
        Ws.append(W)
        bs.append(bh)
        cur = rbm_up(cur, W, bh)
    return Ws, bs


def init_random(sizes, rng, d_in=784):
    """같은 모양을 무작위로 — 비교군. 표준편차는 fan-in 으로 맞춘다."""
    Ws, bs, prev = [], [], d_in
    for h in sizes:
        Ws.append(rng.normal(0, 1.0 / np.sqrt(prev), (prev, h)))
        bs.append(np.zeros(h))
        prev = h
    return Ws, bs


def finetune(Ws, bs, Xtr_, Ytr_, Xte_, yte_, epochs, rng, lr=0.1, mb=MB, momentum=0.9, track_grad=False):
    """로지스틱 은닉층 + 소프트맥스 출력. 지도 역전파. 반환 (시험오차, 학습오차, 층별 기울기 크기)."""
    Ws = [w.copy() for w in Ws]
    bs = [b.copy() for b in bs]
    L = len(Ws)
    Wo = rng.normal(0, 1.0 / np.sqrt(Ws[-1].shape[1]), (Ws[-1].shape[1], 10))
    bo = np.zeros(10)
    vW = [np.zeros_like(w) for w in Ws] + [np.zeros_like(Wo)]
    gnorm0 = None
    n = Xtr_.shape[0]
    for ep in range(epochs):
        idx = rng.permutation(n)
        for s in range(0, n, mb):
            x = Xtr_[idx[s:s + mb]]
            y = Ytr_[idx[s:s + mb]]
            m = x.shape[0]
            acts = [x]
            for l in range(L):
                acts.append(sigmoid(acts[-1] @ Ws[l] + bs[l]))
            z = acts[-1] @ Wo + bo
            z -= z.max(axis=1, keepdims=True)
            p = np.exp(z)
            p /= p.sum(axis=1, keepdims=True)
            d = (p - y) / m
            gW = [None] * L
            gWo = acts[-1].T @ d
            gbo = d.sum(axis=0)
            for l in range(L - 1, -1, -1):
                nxt = Wo if l == L - 1 else Ws[l + 1]
                d = (d @ nxt.T) * acts[l + 1] * (1 - acts[l + 1])
                gW[l] = acts[l].T @ d
                bs[l] -= lr * d.sum(axis=0)
            if track_grad and gnorm0 is None:
                gnorm0 = [float(np.linalg.norm(g)) for g in gW]   # 첫 묶음의 층별 기울기 크기
            for l in range(L):
                vW[l] = momentum * vW[l] - lr * gW[l]
                Ws[l] += vW[l]
            vW[L] = momentum * vW[L] - lr * gWo
            Wo += vW[L]
            bo -= lr * gbo

    def err(Xq, yq):
        a = Xq
        for l in range(L):
            a = sigmoid(a @ Ws[l] + bs[l])
        return float((np.argmax(a @ Wo + bo, axis=1) != yq).mean())

    return err(Xte_, yte_), err(Xtr_, np.argmax(Ytr_, axis=1)), gnorm0


print("구현 준비 완료: train_rbm · pretrain_stack · init_random · finetune")

구현 준비 완료: train_rbm · pretrain_stack · init_random · finetune


### 구현이 맞는지 먼저 확인한다

**둘을 본다.** (가) 미세조정의 기울기가 유한 차분과 맞는가. (나) RBM 의 재구성 오차가 에폭을 따라 내려가는가.

틀린 구현으로 얻은 D 는 아무 뜻이 없으므로 이 칸을 먼저 통과시킨다.

In [4]:
# ── 검증 ────────────────────────────────────────────────────────────────
rng = np.random.default_rng(0)

# (가) 역전파 기울기 대 유한 차분
d_in, h_, nc = 12, 7, 10
Xs = rng.random((6, d_in))
Ys = np.eye(nc)[rng.integers(0, nc, 6)]
W1 = rng.normal(0, .3, (d_in, h_)); b1 = rng.normal(0, .1, h_)
Wo = rng.normal(0, .3, (h_, nc));   bo = rng.normal(0, .1, nc)


def loss_of(W1, b1, Wo, bo):
    a = sigmoid(Xs @ W1 + b1)
    z = a @ Wo + bo
    z = z - z.max(axis=1, keepdims=True)
    p = np.exp(z); p /= p.sum(axis=1, keepdims=True)
    return float(-(Ys * np.log(p + 1e-12)).sum() / Xs.shape[0])


a1 = sigmoid(Xs @ W1 + b1)
z = a1 @ Wo + bo; z -= z.max(axis=1, keepdims=True)
p = np.exp(z); p /= p.sum(axis=1, keepdims=True)
d = (p - Ys) / Xs.shape[0]
gWo = a1.T @ d
d1 = (d @ Wo.T) * a1 * (1 - a1)
gW1 = Xs.T @ d1

eps = 1e-6
num = np.zeros_like(W1)
for i in range(d_in):
    for j in range(h_):
        o = W1[i, j]
        W1[i, j] = o + eps; lp = loss_of(W1, b1, Wo, bo)
        W1[i, j] = o - eps; lm = loss_of(W1, b1, Wo, bo)
        W1[i, j] = o
        num[i, j] = (lp - lm) / (2 * eps)
rel = np.abs(num - gW1).max() / max(np.abs(num).max(), 1e-12)
print("(가) 역전파 기울기 대 유한 차분 · 상대 최대차 %.3e -> %s" % (rel, "맞다" if rel < 1e-5 else "틀렸다"))
assert rel < 1e-5, "기울기 구현이 유한 차분과 맞지 않는다"

# (나) RBM 재구성 오차가 내려가는가
log = []
_W, _bv, _bh = train_rbm(Xtr[:2000], 64, 5, np.random.default_rng(1), k=1, log=log)
print("(나) RBM 재구성 RMSE:", " -> ".join("%.4f" % r for _e, r in log))
print("      내려갔는가:", "그렇다" if log[-1][1] < log[0][1] else "아니다")
assert log[-1][1] < log[0][1], "재구성 오차가 안 내려간다 — 학습률이나 구현을 본다"

(가) 역전파 기울기 대 유한 차분 · 상대 최대차 6.769e-09 -> 맞다
(나) RBM 재구성 RMSE: 0.3227 -> 0.2729 -> 0.2538 -> 0.2355 -> 0.2246
      내려갔는가: 그렇다


## A. RBM 하나를 배운다

논문의 알맹이 중 **실제로 도는 최소 단위**다. 784 픽셀 위에 은닉 256 을 CD-1 로 배우고 셋을 본다 — 에폭별 재구성 오차, 가중치 필터 그림, 자유 에너지가 학습 데이터와 잡음에서 갈리는가.

**자유 에너지**는 RBM 이 그 입력을 얼마나 「그럴듯하다」고 보는지의 값이다. 작은 예시로, 학습한 숫자 이미지에서 낮고 무작위 잡음에서 높으면 모델이 숫자 쪽으로 배워진 것이다.

In [5]:
# ── A. RBM 하나 ─────────────────────────────────────────────────────────
t0 = time.perf_counter()
logA = []
WA, bvA, bhA = train_rbm(Xtr, HIDDEN, RBM_EPOCH, np.random.default_rng(0), k=1, log=logA)
tA = time.perf_counter() - t0


def free_energy(V, W, bv, bh):
    """F(v) = -v.bv - sum_j log(1 + exp(v.W + bh))"""
    x = V @ W + bh
    return -(V @ bv) - np.logaddexp(0, x).sum(axis=1)


fe_data = free_energy(Xte, WA, bvA, bhA)
rngn = np.random.default_rng(7)
fe_noise = free_energy(rngn.random(Xte.shape), WA, bvA, bhA)
fe_shuf = free_energy(np.apply_along_axis(rngn.permutation, 1, Xte[:500]), WA, bvA, bhA)

dfA = pd.DataFrame(logA, columns=["epoch", "recon_rmse"])
dfA["seconds_total"] = round(tA, 2)
save(dfA, "A_rbm_curve.csv")

dfAf = pd.DataFrame({"kind": ["test_digits", "pixel_shuffled", "uniform_noise"],
                     "free_energy_mean": [fe_data.mean(), fe_shuf.mean(), fe_noise.mean()],
                     "free_energy_std": [fe_data.std(), fe_shuf.std(), fe_noise.std()]})
save(dfAf, "A_free_energy.csv")

print()
print("RBM 784 x %d · CD-1 · %d 에폭 · %.1f 초" % (HIDDEN, RBM_EPOCH, tA))
print("  재구성 RMSE %.4f -> %.4f" % (logA[0][1], logA[-1][1]))
print("  자유 에너지 평균 — 시험 숫자 %.1f · 픽셀 섞은 것 %.1f · 균등 잡음 %.1f"
      % (fe_data.mean(), fe_shuf.mean(), fe_noise.mean()))
print("  숫자가 잡음보다 낮은가:", "그렇다" if fe_data.mean() < fe_noise.mean() else "아니다")

저장: A_rbm_curve.csv (15, 8)
저장: A_free_energy.csv (3, 8)

RBM 784 x 256 · CD-1 · 15 에폭 · 31.2 초
  재구성 RMSE 0.2366 -> 0.1241
  자유 에너지 평균 — 시험 숫자 -278.6 · 픽셀 섞은 것 7.8 · 균등 잡음 79.7
  숫자가 잡음보다 낮은가: 그렇다


### 필터 그림

가중치 열 하나가 픽셀 784 개에 붙은 값이라 28 x 28 로 되돌려 볼 수 있다. **획이나 얼룩 모양이 보이면 배워진 것이고, 소금후추 잡음으로 남아 있으면 안 배워진 것이다.**

한글 글꼴이 없는 환경에서 네모로 깨지는 것을 피하려고 **축 라벨은 영문으로 둔다.**

In [6]:
# ── A. 필터와 곡선 그림 ─────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

order = np.argsort(-np.linalg.norm(WA, axis=0))[:64]
fig, axes = plt.subplots(8, 8, figsize=(6.4, 6.4))
for ax, j in zip(axes.ravel(), order):
    w = WA[:, j].reshape(28, 28)
    ax.imshow(w, cmap="gray_r", vmin=-np.abs(w).max(), vmax=np.abs(w).max())
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("A. RBM filters (top 64 by weight norm), CD-1, %d epochs" % RBM_EPOCH, fontsize=11)
fig.tight_layout()
fig.savefig(os.path.join(FIGURES, "A_filters.png"), dpi=130)
plt.close(fig)

fig, ax = plt.subplots(1, 2, figsize=(9.6, 3.4))
ax[0].plot(dfA.epoch, dfA.recon_rmse, marker="o", ms=3)
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("reconstruction RMSE"); ax[0].set_title("A. RBM learning curve")
ax[0].grid(alpha=.3)
ax[1].hist(fe_data, bins=40, alpha=.65, label="test digits")
ax[1].hist(fe_noise, bins=40, alpha=.65, label="uniform noise")
ax[1].set_xlabel("free energy"); ax[1].set_ylabel("count"); ax[1].set_title("A. Free energy")
ax[1].legend(); ax[1].grid(alpha=.3)
fig.tight_layout()
fig.savefig(os.path.join(FIGURES, "A_curve_energy.png"), dpi=130)
plt.close(fig)
print("그림 저장: A_filters.png · A_curve_energy.png")

그림 저장: A_filters.png · A_curve_energy.png


## B. 대조 발산 걸음 수 `n` 을 바꾼다

노트 16절 셋이 적은 자리다 — **논문에 `n` 을 바꿔 본 표가 없다.** 논문 본문은 `n` 을 설명만 하고 MNIST 층별 학습에서 얼마를 썼는지 분명히 적지 않는다(위-아래 단계의 기브스 횟수 3, 6, 10 은 적혀 있다).

`n` 이 클수록 식 5 의 둘째 항이 평형에 가까워지므로 **더 정확한 학습**이어야 하고 **더 비싸야** 한다. 얼마나 그런지 잰다.

In [7]:
# ── B. CD 걸음 수 ───────────────────────────────────────────────────────
rowsB = []
_noise = np.random.default_rng(7).random(Xte.shape)
for k in (1, 3, 10):
    t0 = time.perf_counter()
    lg = []
    Wk, bvk, bhk = train_rbm(Xtr, HIDDEN, RBM_EPOCH, np.random.default_rng(0), k=k, log=lg)
    dt = time.perf_counter() - t0
    fe_d = free_energy(Xte, Wk, bvk, bhk).mean()
    fe_n = free_energy(_noise, Wk, bvk, bhk).mean()
    rowsB.append(dict(cd_steps=k, recon_rmse_first=lg[0][1], recon_rmse_last=lg[-1][1],
                      fe_digits=fe_d, fe_noise=fe_n, fe_gap=fe_n - fe_d, seconds=round(dt, 2)))
    print("CD-%-2d  RMSE %.4f -> %.4f · 자유 에너지 간격 %.1f · %.1f 초"
          % (k, lg[0][1], lg[-1][1], fe_n - fe_d, dt))

dfB = pd.DataFrame(rowsB)
dfB["seconds_vs_cd1"] = (dfB.seconds / dfB.seconds.iloc[0]).round(2)
dfB["rmse_vs_cd1_pct"] = ((dfB.recon_rmse_last / dfB.recon_rmse_last.iloc[0] - 1) * 100).round(2)
dfB["fe_gap_vs_cd1_pct"] = ((dfB.fe_gap / dfB.fe_gap.iloc[0] - 1) * 100).round(2)
save(dfB, "B_cd_steps.csv")
print()
print(dfB[["cd_steps", "recon_rmse_last", "rmse_vs_cd1_pct", "fe_gap", "fe_gap_vs_cd1_pct",
           "seconds", "seconds_vs_cd1"]].to_string(index=False))
print()
print("재구성 오차와 자유 에너지 간격이 서로 다른 것을 잰다 —")
print("  재구성 오차는 '한 걸음 뒤에 얼마나 되돌아오는가' 이고 CD-1 이 바로 그것을 줄이도록 배운다.")
print("  자유 에너지 간격은 '숫자와 잡음을 얼마나 갈라 보는가' 다. k 를 늘릴 때 볼 값은 이쪽이다.")

CD-1   RMSE 0.2366 -> 0.1241 · 자유 에너지 간격 358.3 · 30.9 초
CD-3   RMSE 0.2601 -> 0.1558 · 자유 에너지 간격 387.1 · 51.5 초
CD-10  RMSE 0.3173 -> 0.1896 · 자유 에너지 간격 454.3 · 136.2 초
저장: B_cd_steps.csv (3, 15)

 cd_steps  recon_rmse_last  rmse_vs_cd1_pct     fe_gap  fe_gap_vs_cd1_pct  seconds  seconds_vs_cd1
        1         0.124132             0.00 358.280340               0.00    30.90            1.00
        3         0.155767            25.48 387.111723               8.05    51.54            1.67
       10         0.189631            52.77 454.309422              26.80   136.24            4.41

재구성 오차와 자유 에너지 간격이 서로 다른 것을 잰다 —
  재구성 오차는 '한 걸음 뒤에 얼마나 되돌아오는가' 이고 CD-1 이 바로 그것을 줄이도록 배운다.
  자유 에너지 간격은 '숫자와 잡음을 얼마나 갈라 보는가' 다. k 를 늘릴 때 볼 값은 이쪽이다.


## C. 층별 기울기 — 사전학습이 기울기를 살리는가

역전파 노트북의 실험 E 가 **깊이마다 기울기가 10~12 배씩 줄어든다**를 쟀다. 같은 것을 여기서 **초기화를 바꿔 가며** 잰다.

미세조정 **첫 묶음**에서 층별 기울기 행렬의 크기를 읽는다. 첫 묶음인 이유는 초기화의 효과만 보려는 것이다 — 몇 묶음만 지나도 가중치가 움직여 초기화와 섞인다.

In [8]:
# ── C. 층별 기울기 ──────────────────────────────────────────────────────
rowsC = []
SIZES3 = [HIDDEN, HIDDEN, HIDDEN]
for sd in range(SEEDS):
    rg = np.random.default_rng(100 + sd)
    Wp, bp = pretrain_stack(Xtr, SIZES3, RBM_EPOCH, rg, k=1)
    Wr, br = init_random(SIZES3, np.random.default_rng(200 + sd))
    for tag, (Wl, bl) in (("pretrained", (Wp, bp)), ("random", (Wr, br))):
        _te, _tr, g = finetune(Wl, bl, Xtr, Ytr, Xte, yte, 1, np.random.default_rng(300 + sd), track_grad=True)
        for li, gv in enumerate(g):
            rowsC.append(dict(seed=sd, init=tag, layer=li + 1, grad_norm=gv))
    print("seed %d 끝" % sd)

dfC = pd.DataFrame(rowsC)
save(dfC, "C_layer_grads.csv")
piv = dfC.pivot_table(index="layer", columns="init", values="grad_norm", aggfunc="median")
piv["ratio_pre_over_rand"] = (piv["pretrained"] / piv["random"]).round(3)
print()
print(piv.to_string())
print()
for tag in ("pretrained", "random"):
    v = dfC[dfC.init == tag].pivot_table(index="layer", values="grad_norm", aggfunc="median")["grad_norm"].values
    drops = [v[i] / v[i + 1] for i in range(len(v) - 1)]
    print("%-11s 층을 하나 내려갈 때 기울기가 나뉘는 배수: %s" % (tag, ", ".join("%.1f" % d for d in drops)))

seed 0 끝
seed 1 끝
seed 2 끝
seed 3 끝
seed 4 끝
저장: C_layer_grads.csv (30, 9)

init   pretrained    random  ratio_pre_over_rand
layer                                           
1        0.034210  0.022877                1.495
2        0.047583  0.076803                0.620
3        0.168984  0.338682                0.499

pretrained  층을 하나 내려갈 때 기울기가 나뉘는 배수: 0.7, 0.3
random      층을 하나 내려갈 때 기울기가 나뉘는 배수: 0.3, 0.2


## D. 노트가 꼽은 가장 큰 빈칸 — 사전학습의 몫

**논문에 없는 대조군을 여기서 만든다.** 같은 모양 · 같은 데이터 · 같은 미세조정에, 초기화만 (가) 무작위 (나) 탐욕 사전학습으로 바꾼다. 깊이 1 · 2 · 3 에서 각각 seed 다섯을 돈다.

| | 초기화 | 미세조정 |
|---|---|---|
| 무작위 | fan-in 으로 맞춘 정규분포 | 지도 역전파 30 에폭 |
| 사전학습 | RBM 을 아래층부터 15 에폭씩 | **같은 것** |

**다시 적어 둔다.** 여기의 미세조정은 논문의 위-아래 알고리즘이 아니라 지도 역전파다. 그래서 이 표가 답하는 것은 **「비지도 사전학습이 지도 미세조정을 돕는가」**이고, 「논문의 생성 파이프라인에 탐욕 단계가 필요했나」가 아니다.

In [9]:
# ── D. 사전학습의 몫 ────────────────────────────────────────────────────
rowsD = []
t_all = time.perf_counter()
for depth in (1, 2, 3):
    sizes = [HIDDEN] * depth
    for sd in range(SEEDS):
        rg = np.random.default_rng(1000 + sd)
        t0 = time.perf_counter()
        Wp, bp = pretrain_stack(Xtr, sizes, RBM_EPOCH, rg, k=1)
        t_pre = time.perf_counter() - t0
        Wr, br = init_random(sizes, np.random.default_rng(2000 + sd))
        for tag, (Wl, bl), tp in (("pretrained", (Wp, bp), t_pre), ("random", (Wr, br), 0.0)):
            t1 = time.perf_counter()
            te, tr, _g = finetune(Wl, bl, Xtr, Ytr, Xte, yte, FT_EPOCH,
                                  np.random.default_rng(3000 + sd))
            rowsD.append(dict(depth=depth, seed=sd, init=tag,
                              test_err=te, train_err=tr,
                              pretrain_s=round(tp, 1), finetune_s=round(time.perf_counter() - t1, 1)))
        print("깊이 %d seed %d: 사전학습 %.4f · 무작위 %.4f"
              % (depth, sd, rowsD[-2]["test_err"], rowsD[-1]["test_err"]))

dfD = pd.DataFrame(rowsD)
save(dfD, "D_pretrain_vs_random.csv")
print()
print("D 전체 %.0f 초" % (time.perf_counter() - t_all))

print()
print("깊이 | 초기화     |  시험오차 중앙 | seed %d 개의 변동 폭" % SEEDS)
for depth in (1, 2, 3):
    for tag in ("pretrained", "random"):
        v = dfD[(dfD.depth == depth) & (dfD.init == tag)].test_err.values
        print("  %d  | %-10s | %13.4f | %s" % (depth, tag, np.median(v), span(v)))
    a = dfD[(dfD.depth == depth) & (dfD.init == "pretrained")].test_err.values
    b = dfD[(dfD.depth == depth) & (dfD.init == "random")].test_err.values
    gap = np.median(b) - np.median(a)
    widest = max(a.max() - a.min(), b.max() - b.min())
    verdict = "폭보다 크다" if abs(gap) > widest else "폭 안이다 — 순서를 매기지 않는다"
    print("       차이 %+.4f (무작위 - 사전학습) · 변동 폭 %.4f -> %s" % (gap, widest, verdict))

깊이 1 seed 0: 사전학습 0.0430 · 무작위 0.0560
깊이 1 seed 1: 사전학습 0.0455 · 무작위 0.0565
깊이 1 seed 2: 사전학습 0.0435 · 무작위 0.0565
깊이 1 seed 3: 사전학습 0.0445 · 무작위 0.0615
깊이 1 seed 4: 사전학습 0.0425 · 무작위 0.0595
깊이 2 seed 0: 사전학습 0.0370 · 무작위 0.0530
깊이 2 seed 1: 사전학습 0.0370 · 무작위 0.0585
깊이 2 seed 2: 사전학습 0.0360 · 무작위 0.0590
깊이 2 seed 3: 사전학습 0.0335 · 무작위 0.0615
깊이 2 seed 4: 사전학습 0.0370 · 무작위 0.0555
깊이 3 seed 0: 사전학습 0.0320 · 무작위 0.0545
깊이 3 seed 1: 사전학습 0.0340 · 무작위 0.0605
깊이 3 seed 2: 사전학습 0.0345 · 무작위 0.0555
깊이 3 seed 3: 사전학습 0.0310 · 무작위 0.0575
깊이 3 seed 4: 사전학습 0.0330 · 무작위 0.0635
저장: D_pretrain_vs_random.csv (30, 12)

D 전체 1687 초

깊이 | 초기화     |  시험오차 중앙 | seed 5 개의 변동 폭
  1  | pretrained |        0.0435 | 0.0425~0.0455
  1  | random     |        0.0565 | 0.0560~0.0615
       차이 +0.0130 (무작위 - 사전학습) · 변동 폭 0.0055 -> 폭보다 크다
  2  | pretrained |        0.0370 | 0.0335~0.0370
  2  | random     |        0.0585 | 0.0530~0.0615
       차이 +0.0215 (무작위 - 사전학습) · 변동 폭 0.0085 -> 폭보다 크다
  3  | pretrained |        

In [10]:
# ── D. 그림 ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7.2, 4.0))
for tag, mk in (("pretrained", "o"), ("random", "s")):
    med = [np.median(dfD[(dfD.depth == d) & (dfD.init == tag)].test_err) for d in (1, 2, 3)]
    lo = [dfD[(dfD.depth == d) & (dfD.init == tag)].test_err.min() for d in (1, 2, 3)]
    hi = [dfD[(dfD.depth == d) & (dfD.init == tag)].test_err.max() for d in (1, 2, 3)]
    ax.errorbar([1, 2, 3], med, yerr=[np.array(med) - lo, np.array(hi) - med],
                marker=mk, capsize=4, label=tag)
ax.set_xticks([1, 2, 3]); ax.set_xlabel("hidden layers"); ax.set_ylabel("test error")
ax.set_title("D. greedy pre-training vs random init (bars = seed min-max)")
ax.legend(); ax.grid(alpha=.3)
fig.tight_layout()
fig.savefig(os.path.join(FIGURES, "D_depth.png"), dpi=130)
plt.close(fig)
print("그림 저장: D_depth.png")

그림 저장: D_depth.png


## E. 라벨이 귀할 때

논문 결론이 적는다 — **판별 학습에서는 사례 하나가 라벨을 적는 데 드는 비트만큼만 매개변수를 제약하고, 생성 모델에서는 입력을 적는 데 드는 비트만큼 제약한다.** 그렇다면 라벨이 적을수록 사전학습이 값을 내야 한다.

**사전학습은 라벨을 안 쓰므로 10,000 장 전부를 쓴다.** 미세조정만 라벨 있는 부분집합으로 한다. 그 자리가 이 실험이 재려는 것이다.

In [11]:
# ── E. 라벨 수 ──────────────────────────────────────────────────────────
rowsE = []
SIZES2 = [HIDDEN, HIDDEN]
for sd in range(SEEDS):
    rg = np.random.default_rng(4000 + sd)
    Wp, bp = pretrain_stack(Xtr, SIZES2, RBM_EPOCH, rg, k=1)     # 라벨 없이 전부 쓴다
    Wr, br = init_random(SIZES2, np.random.default_rng(5000 + sd))
    sel_rng = np.random.default_rng(6000 + sd)
    for n_lab in (100, 1000, 10000):
        sel = sel_rng.permutation(N_TRAIN)[:n_lab]
        for tag, (Wl, bl) in (("pretrained", (Wp, bp)), ("random", (Wr, br))):
            te, tr, _g = finetune(Wl, bl, Xtr[sel], Ytr[sel], Xte, yte, FT_EPOCH,
                                  np.random.default_rng(7000 + sd))
            rowsE.append(dict(n_labels=n_lab, seed=sd, init=tag, test_err=te, train_err=tr))
    print("seed %d 끝" % sd)

dfE = pd.DataFrame(rowsE)
save(dfE, "E_label_scarcity.csv")
print()
print("라벨 수 | 초기화     | 시험오차 중앙 | 변동 폭 | 사전학습이 줄인 폭")
for n_lab in (100, 1000, 10000):
    a = dfE[(dfE.n_labels == n_lab) & (dfE.init == "pretrained")].test_err.values
    b = dfE[(dfE.n_labels == n_lab) & (dfE.init == "random")].test_err.values
    widest = max(a.max() - a.min(), b.max() - b.min())
    gap = np.median(b) - np.median(a)
    tail = "폭보다 크다" if abs(gap) > widest else "폭 안이다"
    print(" %6d | pretrained | %12.4f | %s |" % (n_lab, np.median(a), span(a)))
    print(" %6d | random     | %12.4f | %s | %+.4f (%s)" % (n_lab, np.median(b), span(b), gap, tail))

seed 0 끝
seed 1 끝
seed 2 끝
seed 3 끝
seed 4 끝
저장: E_label_scarcity.csv (30, 10)

라벨 수 | 초기화     | 시험오차 중앙 | 변동 폭 | 사전학습이 줄인 폭
    100 | pretrained |       0.2410 | 0.1835~0.2490 |
    100 | random     |       0.7560 | 0.5770~0.8360 | +0.5150 (폭보다 크다)
   1000 | pretrained |       0.0880 | 0.0870~0.0955 |
   1000 | random     |       0.1290 | 0.1255~0.1425 | +0.0410 (폭보다 크다)
  10000 | pretrained |       0.0370 | 0.0360~0.0400 |
  10000 | random     |       0.0565 | 0.0515~0.0605 | +0.0195 (폭보다 크다)


In [12]:
# ── E. 그림 ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7.2, 4.0))
xs = [100, 1000, 10000]
for tag, mk in (("pretrained", "o"), ("random", "s")):
    med = [np.median(dfE[(dfE.n_labels == n) & (dfE.init == tag)].test_err) for n in xs]
    lo = [dfE[(dfE.n_labels == n) & (dfE.init == tag)].test_err.min() for n in xs]
    hi = [dfE[(dfE.n_labels == n) & (dfE.init == tag)].test_err.max() for n in xs]
    ax.errorbar(xs, med, yerr=[np.array(med) - lo, np.array(hi) - med],
                marker=mk, capsize=4, label=tag)
ax.set_xscale("log"); ax.set_xlabel("labelled training images"); ax.set_ylabel("test error")
ax.set_title("E. does pre-training help more when labels are scarce?")
ax.legend(); ax.grid(alpha=.3)
fig.tight_layout()
fig.savefig(os.path.join(FIGURES, "E_labels.png"), dpi=130)
plt.close(fig)
print("그림 저장: E_labels.png")

그림 저장: E_labels.png


## 요약 — 아래 출력을 그대로 `README.md` 의 결과 칸에 옮긴다

**가설마다 맞았는지 틀렸는지를 적는다.** 틀렸으면 틀렸다고 적는다. 변동 폭보다 작은 차이로는 순서를 매기지 않는다.

In [13]:
# ── 요약 ────────────────────────────────────────────────────────────────
print("=" * 84)
print("DBN (2006) 실험 요약 · run_id %s" % RUN_ID)
print("설정: 학습 %d · 시험 %d · 은닉 %d · RBM %d 에폭 · 미세조정 %d 에폭 · seed %d"
      % (N_TRAIN, N_TEST, HIDDEN, RBM_EPOCH, FT_EPOCH, SEEDS))
print("=" * 84)

print()
print("[A] RBM 하나 — H1")
print("  재구성 RMSE %.4f -> %.4f (%d 에폭, %.0f 초)" % (logA[0][1], logA[-1][1], RBM_EPOCH, tA))
print("  자유 에너지: 시험 숫자 %.1f · 픽셀 섞은 것 %.1f · 균등 잡음 %.1f"
      % (fe_data.mean(), fe_shuf.mean(), fe_noise.mean()))
print("  H1 판정:", "맞았다" if (logA[-1][1] < logA[0][1] and fe_data.mean() < fe_noise.mean())
      else "틀렸다 — 필터 그림을 함께 본다")

print()
print("[B] CD 걸음 수 — H2")
for _i, r in dfB.iterrows():
    print("  CD-%-2d RMSE %.4f (CD-1 대비 %+.2f%%) · 자유 에너지 간격 %.1f (%+.2f%%) · %.0f 초 (%.2f 배)"
          % (r.cd_steps, r.recon_rmse_last, r.rmse_vs_cd1_pct, r.fe_gap, r.fe_gap_vs_cd1_pct,
             r.seconds, r.seconds_vs_cd1))
_r1, _r10 = dfB.recon_rmse_last.iloc[0], dfB.recon_rmse_last.iloc[-1]
_g1, _g10 = dfB.fe_gap.iloc[0], dfB.fe_gap.iloc[-1]
_t10 = dfB.seconds_vs_cd1.iloc[-1]
print("  CD-10 은 CD-1 의 %.1f 배 시간이 든다." % _t10)
if _g10 > _g1 and _r10 >= _r1:
    print("  H2 판정: 절반만 맞았다 — 숫자와 잡음을 가르는 힘(자유 에너지 간격)은 늘었는데")
    print("           재구성 오차는 오히려 올라갔다. 둘이 다른 것을 재기 때문이다.")
elif _g10 > _g1:
    print("  H2 판정: 맞았다 — n 을 늘리니 둘 다 좋아졌고 시간이 %.1f 배 들었다." % _t10)
else:
    print("  H2 판정: 틀렸다 — 이 규모에서 n 을 늘려 좋아진 것이 없다.")
    print("           논문이 층별 학습에 작은 n 을 쓴 선택을 뒷받침한다.")

print()
print("[C] 층별 기울기 — H3")
print(piv.to_string())
_r = piv["ratio_pre_over_rand"].values
print("  H3 판정:", "맞았다 — 사전학습 쪽 아래층 기울기가 크다" if _r[0] > 1.0
      else "틀렸다 — 기울기 크기로는 안 갈린다. 이점이 다른 자리에 있다는 뜻이다")

print()
print("[D] 사전학습의 몫 — H4  (노트 16절이 꼽은 가장 큰 빈칸)")
gaps = []
for depth in (1, 2, 3):
    a = dfD[(dfD.depth == depth) & (dfD.init == "pretrained")].test_err.values
    b = dfD[(dfD.depth == depth) & (dfD.init == "random")].test_err.values
    widest = max(a.max() - a.min(), b.max() - b.min())
    gap = np.median(b) - np.median(a)
    gaps.append((gap, widest))
    print("  깊이 %d · 사전학습 %.4f (%s) · 무작위 %.4f (%s) · 차이 %+.4f · 변동 폭 %.4f -> %s"
          % (depth, np.median(a), span(a), np.median(b), span(b), gap, widest,
             "폭보다 크다" if abs(gap) > widest else "폭 안이다"))
_helps = [g > 0 and g > w for g, w in gaps]      # 사전학습이 이기고 그 차이가 변동 폭보다 큰가
_hurts = [g < 0 and -g > w for g, w in gaps]     # 사전학습이 지고 그 차이가 변동 폭보다 큰가
_grow = len(gaps) >= 3 and gaps[2][0] > gaps[0][0]
print("  H4 판정:", end=" ")
if any(_hurts):
    print("틀렸다 — 깊이 %s 에서 사전학습 쪽이 오히려 나쁘고 그 차이가 변동 폭보다 크다"
          % ", ".join(str(d) for d, h in zip((1, 2, 3), _hurts) if h))
elif not any(_helps):
    print("틀렸다 — 세 깊이 모두 차이가 seed 변동 폭 안이다.")
    print("           이 규모에서 사전학습이 값을 못 냈고, 그것이 노트 16절 물음에 대한 답이다.")
elif all(_helps) and _grow:
    print("맞았다 — 세 깊이 모두 사전학습이 이기고 그 차이가 깊이를 따라 커진다")
elif _grow:
    print("절반만 맞았다 — 차이가 깊이를 따라 커지지만 폭보다 큰 깊이는 %s 뿐이다"
          % ", ".join(str(d) for d, h in zip((1, 2, 3), _helps) if h))
else:
    print("절반만 맞았다 — 사전학습이 이기는 깊이는 %s 인데 깊이를 따라 커지지는 않는다"
          % ", ".join(str(d) for d, h in zip((1, 2, 3), _helps) if h))
_chance = dfD[dfD.test_err > 0.85]
if len(_chance):
    print("  주의: 시험 오차가 0.85 를 넘은 칸이 %d 개 있다 — 그 칸은 배우지 못한 것이라"
          % len(_chance))
    print("        「사전학습이 필요 없었다」가 아니라 「둘 다 안 배워졌다」로 읽는다.")

print()
print("[E] 라벨이 귀할 때 — H5")
egaps, esig = [], []
for n_lab in (100, 1000, 10000):
    a = dfE[(dfE.n_labels == n_lab) & (dfE.init == "pretrained")].test_err.values
    b = dfE[(dfE.n_labels == n_lab) & (dfE.init == "random")].test_err.values
    widest = max(a.max() - a.min(), b.max() - b.min())
    gap = np.median(b) - np.median(a)
    egaps.append(gap)
    esig.append(abs(gap) > widest)
    print("  라벨 %5d · 사전학습 %.4f (%s) · 무작위 %.4f (%s) · 차이 %+.4f · 변동 폭 %.4f -> %s"
          % (n_lab, np.median(a), span(a), np.median(b), span(b), gap, widest,
             "폭보다 크다" if abs(gap) > widest else "폭 안이다"))
if not any(esig):
    print("  H5 판정: 말할 수 없다 — 세 자리 모두 차이가 변동 폭 안이라 크기를 견줄 근거가 없다")
elif egaps[-1] < 0 and abs(egaps[-1]) > 0:
    print("  H5 판정: 물음이 성립하지 않는다 — 라벨 10,000 장에서 사전학습 쪽이 이미 나쁘다(%+.4f)."
          % egaps[-1])
    print("           「이점이 커지는가」를 묻기 전에 이점이 있는지부터 안 선다.")
elif egaps[0] > 0 and esig[0] and egaps[0] > egaps[-1]:
    print("  H5 판정: 맞았다 — 라벨 100 장의 차이(%+.4f)가 10,000 장의 차이(%+.4f)보다 크다"
          % (egaps[0], egaps[-1]))
else:
    print("  H5 판정: 틀렸다 — 라벨을 줄여도 차이가 커지지 않는다 (%+.4f -> %+.4f)"
          % (egaps[0], egaps[-1]))

print()
print("=" * 84)
print("적을 때 조심할 것")
print("  1. 미세조정이 논문의 위-아래 알고리즘이 아니라 지도 역전파다.")
print("     그래서 D 는 「비지도 사전학습이 지도 미세조정을 돕는가」에 답하고,")
print("     「논문의 생성 파이프라인에 탐욕 단계가 필요했나」에는 답하지 않는다.")
print("  2. 학습 %d 장은 논문 60,000 장의 %.2f 배이고 은닉이 %d (논문은 500-500-2000) 다."
      % (N_TRAIN, N_TRAIN / 60000, HIDDEN))
print("     절대 오차율을 논문의 1.25% 와 같은 칸에 놓지 않는다.")
print("  3. 변동 폭보다 작은 차이로 순서를 매기지 않는다 — 위의 '폭 안이다' 표시를 그대로 옮긴다.")
print("=" * 84)

DBN (2006) 실험 요약 · run_id 20260920_142811
설정: 학습 10000 · 시험 2000 · 은닉 256 · RBM 15 에폭 · 미세조정 30 에폭 · seed 5

[A] RBM 하나 — H1
  재구성 RMSE 0.2366 -> 0.1241 (15 에폭, 31 초)
  자유 에너지: 시험 숫자 -278.6 · 픽셀 섞은 것 7.8 · 균등 잡음 79.7
  H1 판정: 맞았다

[B] CD 걸음 수 — H2
  CD-1  RMSE 0.1241 (CD-1 대비 +0.00%) · 자유 에너지 간격 358.3 (+0.00%) · 31 초 (1.00 배)
  CD-3  RMSE 0.1558 (CD-1 대비 +25.48%) · 자유 에너지 간격 387.1 (+8.05%) · 52 초 (1.67 배)
  CD-10 RMSE 0.1896 (CD-1 대비 +52.77%) · 자유 에너지 간격 454.3 (+26.80%) · 136 초 (4.41 배)
  CD-10 은 CD-1 의 4.4 배 시간이 든다.
  H2 판정: 절반만 맞았다 — 숫자와 잡음을 가르는 힘(자유 에너지 간격)은 늘었는데
           재구성 오차는 오히려 올라갔다. 둘이 다른 것을 재기 때문이다.

[C] 층별 기울기 — H3
init   pretrained    random  ratio_pre_over_rand
layer                                           
1        0.034210  0.022877                1.495
2        0.047583  0.076803                0.620
3        0.168984  0.338682                0.499
  H3 판정: 맞았다 — 사전학습 쪽 아래층 기울기가 크다

[D] 사전학습의 몫 — H4  (노트 16절이 꼽은 가장 큰 빈칸)
  깊이 1 · 사전학습 0.0435 (0.0425~0.0455) · 무작위 0